# Computation time: benchmarks vs. Evtween
Times every event-generation method on identical inputs, through the same code path the generation runners use (`generate_event_streams.generate_one`). Each method therefore runs exactly as it does when producing `data_test_evt`, with its own interpreter, weights and settings.

**Protocol**
- Every method gets the same clips: the first *k* frames of each selected sequence, with the shared 260×346 centre crop. Clip lengths come from `FRAME_COUNTS` (V2CE needs at least 17 frames).
- Each run is timed end to end with `time.perf_counter()` around the worker process. This includes interpreter start-up, imports, CUDA initialisation, weight loading, inference and writing the output. The process has exited when the clock stops, so asynchronous GPU work is always included.
- One discarded warm-up run per method fills the OS file cache and any CUDA/JIT caches.
- Methods and clips are shuffled in every repeat so drift (thermal throttling, other jobs) does not favour one method.
- Run time grows linearly with clip length, so a line `time = overhead + cost × pairs` is fitted per method:
  - **ms / pair** (slope): steady-state cost per frame interval. Use this to compare speed.
  - **overhead** (intercept): fixed start-up and model-loading cost per call.
  - **RTF** (real-time factor): compute time per second of video. Below 1 means faster than real time.

**Fair comparison.** Run on an otherwise idle machine. Check the device of each method in the environment cell: FracEvent runs on `FRACEVENT_DEVICE` (CPU by default), V2CE needs CUDA, and the other baselines use whatever their worker selects. Report the hardware with the numbers.

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os
import platform
import random
import shutil
import subprocess
import sys
import tempfile
import time
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "benchmarks/run_metric.py").is_file()), Path.cwd())
GEN = root / "benchmarks/event_generation"
sys.path.insert(0, str(GEN))
import generate_event_streams as old
import generate_event_streams_v2 as v2

DATASET = "HQ-EVFI"
SPLIT = "test"
DATA_ROOT = None            # None: $EVTWEEN_DATA, else <lab>/evtween/data (same as the runners)
SEQUENCES = 3               # first N sequences in sorted order, or a list of sequence names
FRAME_COUNTS = (17, 33, 65) # clip lengths in frames; at least two, each >= 17 for V2CE
REPEATS = 3                 # timed runs per (method, clip)
WARMUP = 1                  # discarded runs per method before timing
METHODS = list(old.METHODS) # every baseline, FracEvent and all four Evtween variants
SEED = 0                    # only shuffles the run order
OUT = root / "reports/timing" / DATASET / SPLIT / datetime.now().strftime("%Y%m%d-%H%M%S")

## Environment and preflight
Records the hardware and settings that the numbers depend on, and checks that each method's interpreter, code and weights exist before anything runs.

In [ ]:
def sh(command):
    try:
        return subprocess.run(command, capture_output=True, text=True, timeout=30).stdout.strip()
    except (OSError, subprocess.SubprocessError) as exc:
        return f"unavailable ({exc})"

data = Path(DATA_ROOT).expanduser().resolve() if DATA_ROOT else old.setting("EVTWEEN_DATA", old.LAB / "evtween/data")
assets = old.setting("BENCHMARK_INSTALL", old.LAB / "evtween/evtween-benchmark/benchmarks/event_generation")
try:
    interpreters = json.loads((assets / "interpreters.json").read_text())
except (OSError, ValueError):
    interpreters = {}

ENV_KEYS = ("EVTWEEN_", "FRACEVENT_", "BENCHMARK_", "ALLTEST_", "CUDA_VISIBLE_DEVICES",
            "OMP_NUM_THREADS", "MKL_NUM_THREADS")
cpu = next((line.split(":", 1)[1].strip() for line in sh(["lscpu"]).splitlines()
            if line.startswith("Model name")), platform.processor())
environment = dict(
    started=datetime.now().isoformat(timespec="seconds"),
    host=platform.node(), platform=platform.platform(), cpu=cpu, cpu_count=os.cpu_count(),
    gpu=sh(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"]),
    gpu_busy=sh(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader"]),
    git=sh(["git", "-C", str(root), "rev-parse", "--short", "HEAD"]),
    dataset=DATASET, split=SPLIT, data=str(data), assets=str(assets),
    frame_counts=list(FRAME_COUNTS), repeats=REPEATS, warmup=WARMUP,
    settings={k: v for k, v in os.environ.items() if k.startswith(ENV_KEYS)},
    interpreters=interpreters)

DEVICE = {m: "worker default" for m in METHODS}
DEVICE.update(v2ce="cuda", fracevent=os.getenv("FRACEVENT_DEVICE", "cpu"))
environment["device"] = DEVICE

def preflight(method):
    if method in old.BASELINES:
        if method not in interpreters:
            return f"no interpreter in {assets / 'interpreters.json'}"
        worker = GEN / "v2ce_interval_worker.py" if method == "v2ce" else assets / "worker.py"
        return None if worker.is_file() else f"missing {worker}"
    if method == "fracevent":
        sim = old.setting("FRACEVENT_ROOT", old.LAB / "evtween_bnchmark/FracEvent")
        return None if (sim / "main.py").is_file() else f"missing {sim / 'main.py'}"
    version, mode = old.EVTWEEN[method]
    checkout = (old.setting("EVTWEEN_V52_ROOT", old.LAB / "evtween_bnchmark-v5.2")
                if version == "v5.2" else old.setting("EVTWEEN_V53_ROOT", old.PROJECT))
    weights = old.setting("EVTWEEN_WEIGHTS", old.LAB / "evtween-tides-compare" /
                          "benchmarks/event_generation/checkpoints/evtween_versions")
    ckpt = weights / version / ("model.pt" if mode == "teacher" else "student.pt")
    for path in (checkout / "generate.py", ckpt):
        if not path.is_file():
            return f"missing {path}"
    return None

problems = {m: preflight(m) for m in METHODS}
runnable = [m for m in METHODS if problems[m] is None]
for key in ("host", "cpu", "gpu", "gpu_busy", "git", "data", "assets"):
    print(f"{key:>9}: {environment[key]}")
display(pd.DataFrame({"device": DEVICE, "ready": {m: problems[m] or "ok" for m in METHODS}}))
if not runnable:
    raise RuntimeError("No method is ready; fix the paths above")

## Shared input clips
Prepares each clip once with the runners' own `sequence_info` and `prepare`, so every method reads the same frames, timestamps and (for the teachers) ground-truth events.

In [ ]:
if len(FRAME_COUNTS) < 2 or min(FRAME_COUNTS) < 2:
    raise ValueError("FRAME_COUNTS needs at least two clip lengths of at least two frames")
if "v2ce" in runnable and min(FRAME_COUNTS) < 17:
    raise ValueError("V2CE needs at least 17 frames per clip")
if any(n < 0 for n in (REPEATS - 1, WARMUP)):
    raise ValueError("REPEATS must be >= 1 and WARMUP >= 0")

available = sorted(p for p in (data / DATASET / SPLIT).iterdir() if p.is_dir())
if isinstance(SEQUENCES, int):
    selected = available[:SEQUENCES]
else:
    by_name = {p.name: p for p in available}
    missing = [s for s in SEQUENCES if s not in by_name]
    if missing:
        raise ValueError(f"Unknown sequences: {missing}")
    selected = [by_name[s] for s in SEQUENCES]
if not selected:
    raise ValueError(f"No sequences in {data / DATASET / SPLIT}")

need_gt = any(m.endswith("_teacher") for m in runnable)
OUT.mkdir(parents=True, exist_ok=False)
(OUT / "logs").mkdir()
clips, skipped = [], []
for seq in selected:
    files, ts, _, meta, size = v2.sequence_info(seq)
    for k in sorted(set(FRAME_COUNTS)):
        if k > len(files):
            skipped.append(f"{seq.name}: only {len(files)} frames, needs {k}")
            continue
        times = ts[:k] - ts[0]
        dt = np.diff(times)
        sub = dict(meta, indices=list(range(k)), duration=float(times[-1]),
                   uniform_timestamps=bool(np.allclose(dt, dt.mean(), rtol=1e-3, atol=1e-7)))
        run = OUT / "inputs" / seq.name / f"k{k:04d}"
        v2.prepare(run, seq, files[:k], ts[:k], times, sub, size, need_gt)
        if need_gt and not (run / "ground_truth.npz").is_file():
            skipped.append(f"{seq.name}: no recorded events, teachers will fail on it")
        clips.append(dict(seq=seq.name, k=k, pairs=k - 1, duration_s=float(times[-1]), run=run))

print("\n".join(skipped) or "All clips prepared")
display(pd.DataFrame(clips).drop(columns="run"))
if len({c["k"] for c in clips}) < 2:
    raise ValueError("Need at least two clip lengths to separate overhead from per-pair cost")

## Timing
Each run uses a fresh temporary copy of the clip, and its output is deleted afterwards. Every run is appended to `runs.csv` as it finishes. A method that fails its warm-up is skipped, and the error goes in the table and its log in `logs/`.

In [ ]:
def count_events(path):
    with np.load(path, allow_pickle=False) as z:
        return int(len(z["t"])) if "t" in z.files else np.nan  # V2V outputs voxels

def run_once(method, clip, tag):
    with tempfile.TemporaryDirectory(dir=OUT) as tmp:
        tmp = Path(tmp)
        run, work = tmp / "run", tmp / "work"
        run.mkdir(); work.mkdir()
        for item in clip["run"].iterdir():
            (run / item.name).symlink_to(item)
        log = OUT / "logs" / f"{method}__{clip['seq']}__k{clip['k']}__{tag}.log"
        start = time.perf_counter()
        output = old.generate_one(method, DATASET, run, assets, work, log)
        wall = time.perf_counter() - start
        return wall, count_events(output)

rng = random.Random(SEED)
rows, failed = [], {}
csv = OUT / "runs.csv"
smallest = min(clips, key=lambda c: (c["k"], c["seq"]))

def record(**row):
    rows.append(row)
    pd.DataFrame([row]).to_csv(csv, mode="a", header=not csv.exists(), index=False)

for method in runnable:
    for w in range(WARMUP):
        try:
            wall, _ = run_once(method, smallest, f"warmup{w}")
            print(f"warm-up {method:<22} {wall:8.2f} s", flush=True)
        except Exception as exc:
            failed[method] = str(exc).strip().splitlines()[-1][:300] if str(exc).strip() else type(exc).__name__
            print(f"FAILED  {method:<22} {failed[method]}", flush=True)
            break

active = [m for m in runnable if m not in failed]
total = REPEATS * len(clips) * len(active)
done = 0
for rep in range(REPEATS):
    order = [(c, m) for c in clips for m in active]
    rng.shuffle(order)
    for clip, method in order:
        if method in failed:
            continue
        try:
            wall, events = run_once(method, clip, f"r{rep}")
            ok, error = True, ""
        except Exception as exc:
            wall, events, ok, error = np.nan, np.nan, False, str(exc).strip()[-300:]
        done += 1
        record(method=method, seq=clip["seq"], k=clip["k"], pairs=clip["pairs"],
               duration_s=clip["duration_s"], repeat=rep, wall_s=wall, events=events,
               ok=ok, error=error)
        print(f"[{done:>4}/{total}] {method:<22} {clip['seq']:<24} k={clip['k']:<4} "
              + (f"{wall:8.2f} s" if ok else f"ERROR {error[:80]}"), flush=True)

## Results
Repeats are reduced to their median per (method, sequence, clip length) before fitting, so one slow outlier run does not tilt the line. **±** is the standard error of the slope.

In [ ]:
runs = pd.DataFrame(rows)
good = runs[runs.ok] if len(runs) else runs
EVTWEEN = set(old.EVTWEEN)

def fit(method, group):
    point = group.groupby(["seq", "k"]).agg(pairs=("pairs", "first"), wall=("wall_s", "median"),
                                            duration=("duration_s", "first"), events=("events", "median"))
    x, y = point.pairs.to_numpy(float), point.wall.to_numpy(float)
    if len(np.unique(x)) < 2:
        return None
    if len(x) > 3:
        (slope, intercept), cov = np.polyfit(x, y, 1, cov=True)
        se = float(np.sqrt(cov[0, 0]))
    else:
        slope, intercept = np.polyfit(x, y, 1)
        se = np.nan
    pred = intercept + slope * x
    ss = ((y - y.mean()) ** 2).sum()
    longest = point[point.pairs == x.max()]
    frame_dt = float(np.median(point.duration / point.pairs))
    return dict(
        group="Evtween" if method in EVTWEEN else "Baseline",
        device=DEVICE[method],
        ms_per_pair=1e3 * slope, se_ms=1e3 * se, overhead_s=intercept,
        r2=1 - ((y - pred) ** 2).sum() / ss if ss > 0 else np.nan,
        rtf=slope / frame_dt,
        wall_longest_s=float(longest.wall.median()),
        events_per_s=float((longest.events / longest.wall).median()),
        runs=len(group))

fits = {m: fit(m, g) for m, g in good.groupby("method")} if len(good) else {}
summary = pd.DataFrame({m: f for m, f in fits.items() if f}).T
if len(summary):
    summary = summary.sort_values("ms_per_pair")
    summary.index.name = "method"
for method, reason in failed.items():
    summary.loc[method, ["group", "device", "error"]] = [
        "Evtween" if method in EVTWEEN else "Baseline", DEVICE[method], reason]
for method, reason in problems.items():
    if reason:
        summary.loc[method, ["group", "device", "error"]] = [
            "Evtween" if method in EVTWEEN else "Baseline", DEVICE[method], reason]

per_clip = (good.groupby(["method", "k"]).wall_s.agg(["median", "mean", "std", "count"])
            .unstack("k") if len(good) else pd.DataFrame())

fmt = {"ms_per_pair": "{:.1f}", "se_ms": "±{:.1f}", "overhead_s": "{:.2f}", "r2": "{:.3f}",
       "rtf": "{:.2f}", "wall_longest_s": "{:.2f}", "events_per_s": "{:,.0f}", "runs": "{:.0f}"}
display(summary.style.format(fmt, na_rep="–"))
display(per_clip.round(2))
if len(runs) and (~runs.ok).any():
    print("Failed timed runs:")
    display(runs.loc[~runs.ok, ["method", "seq", "k", "repeat", "error"]])

### Figures
Left: steady-state cost per frame pair, with Evtween variants highlighted. Right: one panel per method with the measured times and the fitted line, which shows where the overhead comes from and whether the linear model holds.

In [ ]:
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
COLOR = {"Evtween": "#2a78d6", "Baseline": "#a3a29c"}
plt.rcParams.update({"axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED,
                     "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False})

ranked = summary.dropna(subset=["ms_per_pair"]) if "ms_per_pair" in summary else summary.iloc[:0]
if len(ranked):
    fig, ax = plt.subplots(figsize=(8, 0.42 * len(ranked) + 1.2))
    y = np.arange(len(ranked))[::-1]
    ax.barh(y, ranked.ms_per_pair.astype(float), height=0.62,
            color=[COLOR[g] for g in ranked.group],
            xerr=ranked.se_ms.astype(float).fillna(0), error_kw=dict(ecolor=MUTED, lw=1, capsize=2))
    values = ranked.ms_per_pair.astype(float)
    ends = values + ranked.se_ms.astype(float).fillna(0)
    pad = 0.015 * ends.max()
    for yi, value, end in zip(y, values, ends):
        ax.text(end + pad, yi, f"{value:,.1f}", va="center", fontsize=8, color=MUTED)
    ax.set_xlim(0, ends.max() * 1.15)
    ax.set_yticks(y, ranked.index)
    ax.set_xlabel("ms per frame pair (lower is faster)")
    ax.grid(axis="x", color=GRID, lw=0.8); ax.set_axisbelow(True)
    ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in COLOR.values()],
              labels=list(COLOR), frameon=False, ncol=2, loc="lower right",
              bbox_to_anchor=(1, 1), fontsize=9)
    ax.set_title(f"Steady-state cost, {DATASET}/{SPLIT}", loc="left", color=INK)
    fig.tight_layout()
    fig.savefig(OUT / "ms_per_pair.png", dpi=200)
    plt.show()

    cols = min(4, len(ranked))
    rows_n = int(np.ceil(len(ranked) / cols))
    fig, axes = plt.subplots(rows_n, cols, figsize=(3.2 * cols, 2.6 * rows_n), squeeze=False)
    for ax, method in zip(axes.flat, ranked.index):
        g = good[good.method == method]
        colour = COLOR[ranked.loc[method, "group"]]
        ax.scatter(g.pairs, g.wall_s, s=16, color=colour, edgecolor="white", lw=0.5, zorder=3)
        xs = np.array([0, g.pairs.max()])
        ax.plot(xs, ranked.loc[method, "overhead_s"] + ranked.loc[method, "ms_per_pair"] / 1e3 * xs,
                color=colour, lw=2)
        ax.set_title(method, fontsize=9, loc="left", color=INK)
        ax.set_xlim(0); ax.set_ylim(0)
        ax.grid(color=GRID, lw=0.8); ax.set_axisbelow(True)
    for ax in axes.flat[len(ranked):]:
        ax.axis("off")
    fig.supxlabel("frame pairs in clip", fontsize=9, color=MUTED)
    fig.supylabel("wall time (s)", fontsize=9, color=MUTED)
    fig.tight_layout()
    fig.savefig(OUT / "wall_vs_pairs.png", dpi=200)
    plt.show()

## Save
Writes the summary, per-clip table and environment next to `runs.csv`, then removes the prepared input clips (set `KEEP_INPUTS = True` to keep them).

In [ ]:
KEEP_INPUTS = False
summary.to_csv(OUT / "summary.csv")
per_clip.to_csv(OUT / "per_clip.csv")
environment["finished"] = datetime.now().isoformat(timespec="seconds")
environment["failed"] = failed
environment["clips"] = [{k: v for k, v in c.items() if k != "run"} for c in clips]
(OUT / "environment.json").write_text(json.dumps(environment, indent=2, default=str))
if not KEEP_INPUTS:
    shutil.rmtree(OUT / "inputs", ignore_errors=True)
print(f"Saved to {OUT}")